# 🛡️ AI vs Real Image Detection - AIForensicNet V4 (Community Forensics CVPR 2025 Edition)

Notebook ini merupakan implementasi model kustom **AIForensicNet V4 (Tri-Stream Forensic CNN)** yang disesuaikan secara khusus untuk dilatih dan dievaluasi menggunakan dataset terobosan terbaru: **[OwensLab/CommunityForensics](https://huggingface.co/datasets/OwensLab/CommunityForensics)** (CVPR 2025, *Andrew Owens Research Lab*).

---

### 📚 Karakteristik Kunci Dataset CommunityForensics (CVPR 2025):
1. **Skala & Keragaman Model Ekstrem**:
   - Berisi **2,7 Juta citra sintetis yang disampling dari 4.803 model generator AI yang berbeda** (34x lipat lebih beragam dibandingkan benchmark media forensik sebelumnya).
   - Mencakup ribuan *fine-tuned community latent diffusion models*, model open-source eksplisit (*StyleGAN 2/3/XL/ADA, BigGAN, DiT, DeepFloyd, Guided Diffusion*), hingga generator komersial mutakhir (*Midjourney V5/V6, DALL-E 2/3, Google Imagen 3, Adobe Firefly 2/3, FLUX.1 dev/schnell, Ideogram*).
2. **Format Data Parquet & Byte Mentah (`image_data`)**:
   - Citra disimpan dalam kolom `image_data` berbentuk byte biner mentah (`bytes`), sehingga didekode secara presisi via `PIL.Image.open(BytesIO(...))`.
3. **Penyelarasan Label (Label Re-Alignment)**:
   - Pada *CommunityForensics*: `1 = Fake (AI-Generated)`, `0 = Real Image`.
   - Pada standar arsitektur proyek ini: `0 = AI-Generated`, `1 = Real Image`.
   - Adapter memetakan secara konsisten: `target_label = 0 if raw_label == 1 else 1`.
4. **Optimasi Khusus Storage Disk Google Colab (Batas 235.7 GB)**:
   - Ukuran total repositori mencapai **1,01 Terabyte** (`systematic` saja 726 GB).
   - Notebook ini menggunakan parameter `data_files` yang **eksklusif hanya mengunduh shard dari subset `PublicEval`** (berpasangan AI + Real seimbang).
   - Disediakan **3 Pilihan Preset Kapasitas Disk**:
     * **`lite`** : 1 Shard (~8.500 citra) $\rightarrow$ **Hanya ~3,2 GB disk** (Training kilat ~10-15 mnt)
     * **`medium`** (Default): 2 Shard (~17.000 citra) $\rightarrow$ **Hanya ~7,4 GB disk** (Training ideal ~25-35 mnt)
     * **`full`** : 6 Shard (51.836 citra) $\rightarrow$ **~26,4 GB disk** (Training komprehensif ~1-1.5 jam, sangat aman di bawah batas 235.7 GB)

In [ ]:
# [Langkah 1] Instalasi dependensi library yang dibutuhkan di Google Colab
!pip install -q datasets torchvision scikit-learn pillow matplotlib tqdm

In [ ]:
import os
import sys
import time
import copy
import json
import shutil
import random
import warnings
from io import BytesIO

warnings.filterwarnings('ignore', message='.*Palette images with Transparency.*')
warnings.filterwarnings('ignore', message='.*xFormers is not available.*')

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, f1_score, confusion_matrix, 
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from datasets import load_dataset, ClassLabel

# ------------------ PENGATURAN SEED & DEVICE ------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cek Kapasitas Disk Colab Saat Ini
total_d, used_d, free_d = shutil.disk_usage('/')
print('=' * 75)
if torch.cuda.is_available():
    print(f"🚀 Akselerasi GPU Terdeteksi : {device} ({torch.cuda.get_device_name(0)})")
    print(f"💾 Total Alokasi VRAM        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print(f"⚠️  Berjalan pada CPU (Disarankan aktifkan GPU T4/A100 di menu Runtime -> Change runtime type)")
print(f"💽 Kapasitas Disk Colab      : {used_d / (1024**3):.1f} GB terpakai / {total_d / (1024**3):.1f} GB total (Sisa Bebas: {free_d / (1024**3):.1f} GB)")
print('=' * 75)

## 🧹 [PENTING] Sel Pembersihan Cache Darurat (Jika Disk Sebelumnya Penuh)
Jika pada percobaan sebelumnya Anda sempat menjalankan script yang mencoba mengunduh repositori 1TB hingga disk Colab 235.7 GB hampir habis, **hapus tanda komentar (`#`) pada sel di bawah ini dan jalankan** untuk me-reset sisa disk kembali lega.

In [ ]:
# Jalankan baris di bawah ini jika disk Colab Anda terlanjur terisi sisa download lama:
# !rm -rf ~/.cache/huggingface/hub/datasets--OwensLab--CommunityForensics*
# !rm -rf ~/.cache/huggingface/datasets/OwensLab___community_forensics*
# print("Cache lama berhasil dibersihkan!")

## 🧠 2. Definisi Arsitektur Model AIForensicNet V4 (Self-Contained)
Arsitektur ini didesain murni dari nol (*from scratch*) menggabungkan 3 ranah forensik (*Tri-Stream*):
1. **Multi-Scale Static Residual + NPR (15 Kanal)**: Laplacian 3x3, Laplacian 5x5, DoG 5x5, Sobel Magnitude, dan *Neighboring Pixel Relationships* (CVPR 2024).
2. **2D-FFT Spectral Branch (32 Kanal)**: Mengukur anomali simetri radial pada spektrum Fourier 2D.
3. **Spatial RGB Branch (32 Kanal)**: Mengukur koherensi semantik visual dan pencahayaan alami.
4. **MultiStatPool2d**: Menggabungkan mean dan standar deviasi aktivasi spasial menjadi representasi 1024-dimensi.

In [ ]:
# --- SQUEEZE-AND-EXCITATION ATTENTION ---
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, reduced, bias=False),
            nn.GELU(),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w

# --- FORENSIC RESIDUAL BLOCK ---
class ForensicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.dwconv = nn.Conv2d(in_c, in_c, kernel_size=7, stride=stride, padding=3, groups=in_c, bias=False)
        self.norm = nn.BatchNorm2d(in_c)
        self.pwconv1 = nn.Conv2d(in_c, out_c * 2, kernel_size=1)
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(out_c * 2, out_c, kernel_size=1)
        self.se = SqueezeExcitation(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        res = self.shortcut(x)
        out = self.dwconv(x)
        out = self.norm(out)
        out = self.pwconv1(out)
        out = self.act(out)
        out = self.pwconv2(out)
        out = self.se(out)
        return out + res

# --- NPR MODULE (CVPR 2024) ---
class NPRModule(nn.Module):
    def forward(self, x):
        down = F.interpolate(x, scale_factor=0.5, mode='nearest')
        up = F.interpolate(down, size=x.shape[2:], mode='nearest')
        return x - up

# --- 2D-FFT SPECTRAL BRANCH (32 KANAL) ---
class FFTSpectralBranch(nn.Module):
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.BatchNorm2d(16)
        self.act1 = nn.GELU()
        self.conv2 = nn.Conv2d(16, out_channels, kernel_size=3, stride=2, padding=1, bias=False)
        self.norm2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
    def forward(self, x):
        fft = torch.fft.rfft2(x, norm='ortho')
        mag = torch.log(torch.abs(fft) + 1e-6)
        mag_interp = F.interpolate(mag, size=x.shape[2:], mode='bilinear', align_corners=False)
        feat = self.act1(self.norm1(self.conv1(mag_interp)))
        feat = self.act2(self.norm2(self.conv2(feat)))
        return feat

# --- MULTI-SCALE FORENSIC FILTER BANK V4 (15 KANAL) ---
class MultiScaleForensicFilterBankV4(nn.Module):
    def __init__(self):
        super().__init__()
        lap3 = torch.tensor([[-1., -1., -1.], [-1., 8., -1.], [-1., -1., -1.]], dtype=torch.float32)
        lap5 = torch.tensor([[0., 0., -1., 0., 0.], [0., -1., -2., -1., 0.], [-1., -2., 16., -2., -1.], [0., -1., -2., -1., 0.], [0., 0., -1., 0., 0.]], dtype=torch.float32)
        dog = torch.tensor([[-1., -3., -4., -3., -1.], [-3., 0., 6., 0., -3.], [-4., 6., 20., 6., -4.], [-3., 0., 6., 0., -3.], [-1., -3., -4., -3., -1.]], dtype=torch.float32) / 16.0
        sobel_h = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], dtype=torch.float32)
        sobel_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32)
        k3 = torch.stack([lap3, sobel_h, sobel_v]).unsqueeze(1).repeat(3, 1, 1, 1)
        k5 = torch.stack([lap5, dog]).unsqueeze(1).repeat(3, 1, 1, 1)
        self.register_buffer('k3', k3)
        self.register_buffer('k5', k5)
        self.npr = NPRModule()
    def forward(self, x):
        f3 = F.conv2d(x, self.k3, padding=1, groups=3)
        f5 = F.conv2d(x, self.k5, padding=2, groups=3)
        sh = f3[:, 1::3, :, :]
        sv = f3[:, 2::3, :, :]
        smag = torch.sqrt(sh ** 2 + sv ** 2 + 1e-6)
        lap3_out = f3[:, 0::3, :, :]
        lap5_out = f5[:, 0::2, :, :]
        dog_out = f5[:, 1::2, :, :]
        npr_out = self.npr(x)
        return torch.cat([lap3_out, lap5_out, dog_out, smag, npr_out], dim=1)

# --- MULTI-STAT GLOBAL POOLING (MEAN + STD) ---
class MultiStatPool2d(nn.Module):
    def forward(self, x):
        mean = torch.mean(x, dim=(2, 3))
        var = torch.var(x, dim=(2, 3), unbiased=False)
        std = torch.sqrt(var + 1e-6)
        return torch.cat([mean, std], dim=1)

# --- ARSITEKTUR UTAMA AIForensicNet V4 ---
class AIForensicNetV4(nn.Module):
    def __init__(self, num_classes=2, dropout_p=0.25):
        super().__init__()
        self.forensic_bank = MultiScaleForensicFilterBankV4()
        self.forensic_stem = nn.Sequential(
            nn.Conv2d(15, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.spectral_branch = FFTSpectralBranch(in_channels=3, out_channels=32)
        self.rgb_stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(96, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU()
        )
        self.stage1 = nn.Sequential(ForensicBlock(64, 64), ForensicBlock(64, 64))
        self.stage2 = nn.Sequential(ForensicBlock(64, 128, stride=2), ForensicBlock(128, 128))
        self.stage3 = nn.Sequential(ForensicBlock(128, 256, stride=2), ForensicBlock(256, 256))
        self.stage4 = nn.Sequential(ForensicBlock(256, 512, stride=2), ForensicBlock(512, 512))
        self.pool = MultiStatPool2d()
        self.head = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(512 * 2, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(p=0.15),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        forensic_signals = self.forensic_bank(x)
        feat_forensic = self.forensic_stem(forensic_signals)
        feat_spectral = self.spectral_branch(x)
        feat_rgb = self.rgb_stem(x)
        tri_feat = torch.cat([feat_forensic, feat_spectral, feat_rgb], dim=1)
        out = self.fusion(tri_feat)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        pooled = self.pool(out)
        return self.head(pooled)

# Verifikasi forward pass model
dummy_model = AIForensicNetV4(num_classes=2).to(device)
dummy_in = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    dummy_out = dummy_model(dummy_in)
total_params = sum(p.numel() for p in dummy_model.parameters())
print(f"✅ AIForensicNet V4 Terverifikasi: Output Shape = {dummy_out.shape}, Total Parameter = {total_params:,} (~{total_params/1e6:.2f}M)")

## 🛠️ 3. Forensik Data Pipeline & Adapter Dataset CommunityForensics
- **`RandomJPEGCompression`**: Melatih model agar resistan terhadap artefak kompresi media sosial.
- **`NativeResolutionCrop`**: Mempertahankan resolusi asli piksel tanpa blur downsampling global.
- **`CommunityForensicsDataset`**: Mendekode `image_data` dari format byte biner mentah dan memetakan label `1 (Fake) -> 0 (AI)` serta `0 (Real) -> 1 (Real)`.

In [ ]:
class RandomJPEGCompression:
    def __init__(self, quality_range=(60, 95), p=0.45):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):
        if random.random() < self.p:
            quality = random.randint(*self.quality_range)
            buf = BytesIO()
            img.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            return Image.open(buf).convert('RGB')
        return img

class NativeResolutionCrop:
    def __init__(self, size=224, is_train=True):
        self.size = size
        self.is_train = is_train

    def __call__(self, img):
        w, h = img.size
        if min(w, h) < self.size:
            scale = float(self.size) / float(min(w, h))
            new_w, new_h = int(round(w * scale)), int(round(h * scale))
            img = img.resize((new_w, new_h), Image.Resampling.BILINEAR)
            w, h = img.size

        if self.is_train:
            left = random.randint(0, w - self.size)
            top = random.randint(0, h - self.size)
        else:
            left = (w - self.size) // 2
            top = (h - self.size) // 2

        return img.crop((left, top, left + self.size, top + self.size))

class CommunityForensicsDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if "image_data" in item and item["image_data"] is not None:
            raw_bytes = item["image_data"]
            image = Image.open(BytesIO(raw_bytes))
        elif "image" in item and item["image"] is not None:
            image = item["image"]
            if not isinstance(image, Image.Image):
                image = Image.open(BytesIO(image))
        else:
            raise KeyError("Tidak ditemukan kolom 'image_data' atau 'image' pada sampel data.")

        if image.mode == 'P' and 'transparency' in image.info:
            image = image.convert('RGBA')
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Penyelarasan label: CommunityForensics (1: Fake -> 0: AI, 0: Real -> 1: Real)
        raw_label = int(item["label"])
        target_label = 0 if raw_label == 1 else 1

        if self.transform:
            image = self.transform(image)

        return image, target_label

def get_v4_transforms(img_size=224):
    train_transforms = transforms.Compose([
        RandomJPEGCompression(quality_range=(60, 95), p=0.45),
        NativeResolutionCrop(size=img_size, is_train=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.4))], p=0.15),
        transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.06),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_transforms = transforms.Compose([
        NativeResolutionCrop(size=img_size, is_train=False),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transforms, eval_transforms

print("✅ Pipeline Augmentasi & Adapter Dataset Siap!")

## 📦 4. Pemuatan Dataset Terkontrol (Aman dari Batas Disk 235.7 GB)
Pilih salah satu preset sesuai kebutuhan Anda:
- **`lite`** : 1 Shard Parquet (~8.500 citra, memakan disk **~3,2 GB**). Waktu latih ~10-15 menit.
- **`medium`** (Default): 2 Shard Parquet (~17.000 citra, memakan disk **~7,4 GB**). Waktu latih ~25-35 menit.
- **`full`** : 6 Shard Parquet (51.836 citra, memakan disk **~26,4 GB**). Waktu latih ~1-1.5 jam.

Parameter `data_files` menjamin Hugging Face **hanya mengunduh shard yang dipilih**, dan tidak pernah menyentuh folder `systematic` (726 GB).

In [ ]:
# ================= PILIHAN PRESET DISK & SAMPEL =================
# Pilihan: "lite" (1 Shard ~3.2GB), "medium" (2 Shard ~7.4GB), "full" (6 Shard ~26.4GB)
DISK_PRESET = "medium"

if DISK_PRESET == "lite":
    target_data_files = ["data/publicEval/HFCF_PublicEval_0.parquet"]
    target_samples = 8000
elif DISK_PRESET == "medium":
    target_data_files = [
        "data/publicEval/HFCF_PublicEval_0.parquet",
        "data/publicEval/HFCF_PublicEval_1.parquet"
    ]
    target_samples = 16000
else: # full
    target_data_files = [f"data/publicEval/HFCF_PublicEval_{i}.parquet" for i in range(6)]
    target_samples = -1

BATCH_SIZE = 32
NUM_WORKERS = 2

print(f"📦 Memuat Hugging Face Dataset dengan Preset '{DISK_PRESET}'...")
print("Target Parquet File:")
for f in target_data_files:
    print(f"  ├─ {f}")

raw_dataset = load_dataset(
    "OwensLab/CommunityForensics",
    data_files=target_data_files,
    split="train",
    verification_mode="no_checks"
)

total_loaded = len(raw_dataset)
print(f"\nBerhasil memuat {total_loaded:,} citra secara aman tanpa memenuhi disk!")

from datasets import ClassLabel
if not isinstance(raw_dataset.features["label"], ClassLabel):
    raw_dataset = raw_dataset.cast_column("label", ClassLabel(num_classes=2, names=["0", "1"]))

# Subsampling terstratifikasi jika diset target_samples
if target_samples > 0 and target_samples < total_loaded:
    print(f"⚡ Mengambil subset terstratifikasi {target_samples:,} citra untuk efisiensi komputasi...")
    sub_split = raw_dataset.train_test_split(train_size=target_samples, stratify_by_column="label", seed=42)
    working_data = sub_split["train"]
else:
    working_data = raw_dataset

raw_labels = working_data["label"]
ai_count = sum(1 for l in raw_labels if l == 1)
real_count = sum(1 for l in raw_labels if l == 0)
print(f"Distribusi Sampel Digunakan:")
print(f"├─ Total Citra           : {len(working_data):,}")
print(f"├─ AI-Generated (Asli 1) : {ai_count:,} ({ai_count/len(working_data)*100:.1f}%) -> Label Model: 0")
print(f"└─ Real Image (Asli 0)   : {real_count:,} ({real_count/len(working_data)*100:.1f}%) -> Label Model: 1")

# 3-Way Stratified Split (80% Train, 10% Val, 10% Test)
split1 = working_data.train_test_split(test_size=0.20, stratify_by_column="label", seed=42)
train_hf = split1["train"]
val_test_hf = split1["test"]

split2 = val_test_hf.train_test_split(test_size=0.50, stratify_by_column="label", seed=42)
val_hf = split2["train"]
test_hf = split2["test"]

print(f"\nPembagian Data 3-Way:")
print(f"├─ Train (80%)      : {len(train_hf):,} citra")
print(f"├─ Validation (10%) : {len(val_hf):,} citra (seleksi checkpoint)")
print(f"└─ Test Murni (10%) : {len(test_hf):,} citra (evaluasi final)")

train_transforms, eval_transforms = get_v4_transforms(img_size=224)
train_ds = CommunityForensicsDataset(train_hf, transform=train_transforms)
val_ds = CommunityForensicsDataset(val_hf, transform=eval_transforms)
test_ds = CommunityForensicsDataset(test_hf, transform=eval_transforms)

use_pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_pin, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_pin, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_pin, persistent_workers=True)
print("✅ DataLoaders Berhasil Disiapkan!")

## 🖼️ 5. Visualisasi Sampel Dataset Forensik
Mari kita tampilkan sampel visual acak dari dataset CommunityForensics lengkap dengan label dan model generatornya.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

indices = random.sample(range(len(train_hf)), 8)
for i, idx in enumerate(indices):
    item = train_hf[idx]
    img = Image.open(BytesIO(item["image_data"])).convert("RGB")
    lbl = "AI-Generated (0)" if item["label"] == 1 else "Real Image (1)"
    model_name = item.get("model_name", "Unknown")[:20]
    
    axes[i].imshow(img)
    axes[i].set_title(f"{lbl}\nModel: {model_name}\nRes: {img.size[0]}x{img.size[1]}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 🚀 6. Pelatihan AIForensicNet V4 dengan Cosine Annealing, AMP & Label Smoothing

In [ ]:
EPOCHS = 15
LR = 5e-4
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.05
ACCUM_STEPS = 1
OUTPUT_CHECKPOINT = "best_aiforensicnet_v4_cf.pth"

model = AIForensicNetV4(num_classes=2, dropout_p=0.25).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
eval_criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

warmup_epochs = min(2, max(1, EPOCHS // 4))
cosine_epochs = max(1, EPOCHS - warmup_epochs)

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return float(epoch + 1) / float(max(1, warmup_epochs))
    else:
        progress = float(epoch - warmup_epochs) / float(max(1, cosine_epochs))
        return max(1e-3, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0.0
best_weights = copy.deepcopy(model.state_dict())

print(f"🚀 Memulai Pelatihan {EPOCHS} Epochs pada {device} ...")
start_time = time.time()

for epoch in range(EPOCHS):
    ep_start = time.time()
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    train_bar = tqdm(train_loader, desc=f"Ep [{epoch+1:02d}/{EPOCHS:02d}] Train", leave=False)
    for step, (images, labels) in enumerate(train_bar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += images.size(0)

        train_bar.set_postfix({"loss": f"{loss.item()*ACCUM_STEPS:.4f}", "acc": f"{correct_train/total_train*100:.2f}%"})

    train_loss = running_loss / total_train
    train_acc = correct_train / total_train * 100.0

    # Evaluasi Validasi
    model.eval()
    val_loss = 0.0
    v_preds, v_targets = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = eval_criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            v_preds.extend(preds.cpu().numpy())
            v_targets.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_hf)
    val_acc = (np.array(v_preds) == np.array(v_targets)).mean() * 100.0
    val_f1 = f1_score(v_targets, v_preds, average='macro')

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    flag = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_weights = copy.deepcopy(model.state_dict())
        torch.save(best_weights, OUTPUT_CHECKPOINT)
        flag = "⭐ [CHECKPOINT TERSIMPAN]"

    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] ({time.time()-ep_start:.1f}s) | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% F1: {val_f1:.4f} {flag}")

print(f"\nPelatihan tuntas dalam {(time.time()-start_time)/60:.2f} menit! Best Macro F1: {best_val_f1:.4f}")

## 📈 7. Visualisasi Kurva Metrik Pelatihan

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', marker='s')
plt.title('Kurva Loss (Pelatihan vs Validasi)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history['train_acc'], label='Train Acc (%)', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_acc'], label='Val Acc (%)', marker='s')
plt.plot(range(1, EPOCHS + 1), [f * 100 for f in history['val_f1']], label='Val F1 (x100)', linestyle='--')
plt.title('Kurva Akurasi & F1-Score', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Persentase (%)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

## 🔬 8. Evaluasi Akhir pada Test Set Murni (Held-Out Test Set)
Himpunan uji murni (10% data) yang sama sekali belum pernah disentuh selama pelatihan dievaluasi satu kali untuk mendapatkan metrik performa akhir yang objektif dan valid secara akademis.

In [ ]:
# Muat checkpoint bobot terbaik
model.load_state_dict(best_weights)
model.eval()

t_preds, t_targets, t_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Menguji Data Test"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        t_preds.extend(preds.cpu().numpy())
        t_targets.extend(labels.cpu().numpy())
        t_probs.extend(probs[:, 1].cpu().numpy())

t_preds = np.array(t_preds)
t_targets = np.array(t_targets)
t_probs = np.array(t_probs)

test_acc = (t_preds == t_targets).mean() * 100.0
test_f1 = f1_score(t_targets, t_preds, average='macro')
test_auc = roc_auc_score(t_targets, t_probs) * 100.0

print('=' * 75)
print(f"🎯 Akurasi Data Uji (Test Accuracy) : {test_acc:.2f}%")
print(f"🎯 Macro F1-Score Data Uji          : {test_f1:.4f}")
print(f"🎯 Area Under ROC (ROC-AUC)         : {test_auc:.2f}%")
print('=' * 75)
print("\n📋 CLASSIFICATION REPORT:")
print(classification_report(t_targets, t_preds, target_names=['AI-Generated (0)', 'Real Image (1)'], digits=4))

# Plot Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(t_targets, t_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['AI (0)', 'Real (1)'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Test Set)', fontweight='bold')

fpr, tpr, _ = roc_curve(t_targets, t_probs)
axes[1].plot(fpr, tpr, label=f'AIForensicNet V4 (AUC = {test_auc:.2f}%)', color='darkorange', lw=2)
axes[1].plot([0, 1], [0, 1], color='navy', linestyle='--', lw=1.5)
axes[1].set_title('ROC Curve (Test Set)', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 🧪 9. Uji Inferensi Interaktif pada Citra Kustom (dengan TTA)

In [ ]:
def predict_image(img, model, device):
    eval_transforms = transforms.Compose([
        NativeResolutionCrop(size=224, is_train=False),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    if img.mode != 'RGB':
        img = img.convert('RGB')
        
    t_orig = eval_transforms(img).unsqueeze(0).to(device)
    t_flip = torch.flip(t_orig, dims=[3])
    
    model.eval()
    with torch.no_grad():
        p_orig = torch.softmax(model(t_orig), dim=1)[0]
        p_flip = torch.softmax(model(t_flip), dim=1)[0]
        probs = (p_orig + p_flip) / 2.0
        
    ai_prob = probs[0].item() * 100.0
    real_prob = probs[1].item() * 100.0
    
    if ai_prob >= 80.0:
        verdict = f"🔴 AI-Generated Image ({ai_prob:.1f}%)"
    elif real_prob >= 80.0:
        verdict = f"🟢 Real Image ({real_prob:.1f}%)"
    else:
        verdict = f"🟡 Terdeteksi Sebagian Dimodifikasi AI (AI: {ai_prob:.1f}%, Real: {real_prob:.1f}%)"
        
    return verdict, ai_prob, real_prob

# Uji pada salah satu citra test set
sample_item = test_hf[0]
sample_img = Image.open(BytesIO(sample_item["image_data"]))
verdict, ai_p, real_p = predict_image(sample_img, model, device)

plt.figure(figsize=(5, 5))
plt.imshow(sample_img)
plt.title(f"Hasil Prediksi:\n{verdict}\nGround Truth: {'AI (0)' if sample_item['label']==1 else 'Real (1)'}", fontsize=11)
plt.axis('off')
plt.show()